### Pydantic 모델

- 정의된 데이터만 전송되도록 요청 바디 검증


In [ ]:
from datetime import datetime

from pydantic import BaseModel, PositiveInt, ValidationError, PositiveFloat


class User(BaseModel):
    id: int
    name: str = "John Doe"
    signup_ts: datetime | None
    tastes: dict[str, PositiveInt]


external_data = {
    "id": 123,
    "signup_ts": "2019-06-01 12:22",
    "tastes": {
        "wine": 9,
        b"cheese": 7,
        "cabbage": "1",
    },
}

user = User(**external_data)

print(user.id)

print(user.model_dump())  # model의 필드와 값을 dict 형태로 변환

123
{'id': 123, 'name': 'John Doe', 'signup_ts': datetime.datetime(2019, 6, 1, 12, 22), 'tastes': {'wine': 9, 'cheese': 7, 'cabbage': 1}}


In [ ]:
external_data = {"id": "not an int", "tastes": {}}
try:
    User(**external_data)
except ValidationError as e:
    print(e.errors())

[{'type': 'int_parsing', 'loc': ('id',), 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'not an int', 'url': 'https://errors.pydantic.dev/2.13/v/int_parsing'}, {'type': 'missing', 'loc': ('signup_ts',), 'msg': 'Field required', 'input': {'id': 'not an int', 'tastes': {}}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [ ]:
# Fruit : name - str color = red or green, weight - positivefloat bazam - dict[key:str , value:[(tuple-int,bool,float)]]
from typing import Annotated, Literal
from annotated_types import Gt


# Gt(greater than)
class Fruit(BaseModel):
    name: str
    color: Literal["red", "green"]
    weight: Annotated[float, Gt(0)]
    bazam: dict[str, list[tuple[int, bool, float]]]

In [ ]:
fruit = Fruit(name="Apple", color="red", weight=4.2, bazam={"foobar": [(1, True, 0.1)]})

In [ ]:
fruit2 = Fruit(name="Apple", color="red", weight=4.2)
fruit2

In [ ]:
# Book code(4자리), category(소설, 과학, 역사, 경제, 자기계발, 컴퓨터), title-str, author-str, price- int, >0

from pydantic import StringConstraints


class Book(BaseModel):
    code: Annotated[str, StringConstraints(min_length=4, max_length=4)]
    category: Literal[
        "novel", "science", "history", "economy", "computer", "self-improvement"
    ]
    title: str
    author: str
    price: Annotated[int, Gt(0)]

In [16]:
try:
    Book(code="1211", category="history", price=12100)
except ValidationError as e:
    print(e.errors())

[{'type': 'missing', 'loc': ('title',), 'msg': 'Field required', 'input': {'code': '1211', 'category': 'history', 'price': 12100}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}, {'type': 'missing', 'loc': ('author',), 'msg': 'Field required', 'input': {'code': '1211', 'category': 'history', 'price': 12100}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [ ]:
# User-id:int, name:str, age:14~100 , email, role="user" or "admin"

from pydantic import Field


class User(BaseModel):
    id: int
    name: str
    age: Annotated[int, Field(ge=14, le=100)]
    email: str
    role: Literal["user", "admin"]

In [ ]:
try:
    User(id=1, name="hong", age=101, email="hong@gmail.com", role="user")
except ValidationError as e:
    print(e.errors())

[{'type': 'less_than_equal', 'loc': ('age',), 'msg': 'Input should be less than or equal to 100', 'input': 101, 'ctx': {'le': 100}, 'url': 'https://errors.pydantic.dev/2.13/v/less_than_equal'}]


In [ ]:
# Product : code-문자, name - 문자, category - food, clothes, electronics , price - 정수 >0, stock - 정수 >=0
from annotated_types import Ge, Le


class Product(BaseModel):
    code: str
    name: str
    category: Literal["food", "clothes", "electronics"]
    price: Annotated[float, Gt(0)]
    stock: Annotated[int, Ge(0)]

In [ ]:
try:
    Product(
        code="DCAQ221", name="Power Outlet", category="electronics", price=5.99, stock=9
    )
except ValidationError as e:
    print(e.errors())

In [ ]:
class Item(BaseModel):
    item: str
    status: str


class Cart(BaseModel):
    id: int
    item: Item

In [ ]:
try:
    cart = Cart(id=1, item=Item(item="바지", status="판매"))
except ValidationError as e:
    print(e.errors())

In [ ]:
class OrderItem(BaseModel):
    product_name: str
    quantity: Annotated[int, Gt(0)]
    price: Annotated[int, Gt(0)]


class Order(BaseModel):
    order_id: int
    customer: str
    items: list[OrderItem]

In [ ]:
order = Order(
    order_id=1001,
    customer="jason",
    items=[
        {"product_name": "laptop", "quantity": 1, "price": 1210000},
        {"product_name": "mouse", "quantity": 1, "price": 28900},
    ],
)

In [ ]:
# Comment : author-str  , content- str
# post - [id - int, title- str author - str, content - str, comments - comment]


class Comment(BaseModel):
    author: str
    content: str


class Post(BaseModel):
    id: int
    title: str
    author: str
    content: str
    comments: Comment

In [ ]:
try:
    post = Post(
        id=1,
        title="SundayPost",
        author="SundaySunny",
        content="SundayWeather",
        comments={"author": "weatherlover", "content": "good forecast"},
    )
except ValidationError as e:
    print(e.errors())